# LoRA fine-tuning: стандартный loss HF против CCE

Аналог рис. 4 статьи: кривые loss при обучении с CCE и без неё должны совпадать.

**Порядок запуска.** `cce_patch` подменяет forward модели во всём процессе Python, поэтому прогоны идут в разных ядрах:
1. `LOSS_IMPL = "hf"` → Run All.
2. **Restart kernel**, `LOSS_IMPL = "cce"` → Run All.
3. Раздел «Сравнение прогонов» читает только CSV из `results/`, его можно запускать в любом ядре.

Если не хватает памяти: уменьшить `batch_size` и увеличить `grad_accum` (эффективный батч тот же) или включить `gradient_checkpointing`.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

DEVICE = "cuda"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

## 1. Настройки

In [ ]:
from finetune import FinetuneConfig

LOSS_IMPL = "hf"   # "hf" или "cce"; после смены — перезапустить ядро
MODEL_NAME = "google/gemma-2-2b-it"
DATA_CSV = "data/alpaca.csv"
MAX_LENGTH = 512
SEED = 0

cfg = FinetuneConfig(
    model_name=MODEL_NAME,
    loss_impl=LOSS_IMPL,
    cce_impl="cce",
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    optimizer="adamw",
    lr=2e-4,
    warmup_steps=20,
    num_steps=300,
    batch_size=4,
    grad_accum=2,            # эффективный батч = 8 примеров
    max_grad_norm=1.0,
    gradient_checkpointing=False,
    seed=SEED,
    log_every=1,
    log_path=str(RESULTS / f"finetune_{LOSS_IMPL}.csv"),
)
cfg

## 2. Модель и данные

In [ ]:
from finetune import count_trainable_parameters, load_model_for_finetune

model, tokenizer = load_model_for_finetune(cfg, device=DEVICE)
trainable, total = count_trainable_parameters(model)
print(f"Обучаемых параметров: {trainable / 1e6:.1f}M из {total / 1e6:.0f}M ({100 * trainable / total:.2f}%)")

In [ ]:
from preprocessing import count_loss_tokens, load_examples, make_train_dataloader, tokenize_examples

examples = load_examples(DATA_CSV)
tokenized = tokenize_examples(tokenizer, examples, max_length=MAX_LENGTH, mask_prompt=True)
dataloader = make_train_dataloader(tokenized, batch_size=cfg.batch_size, pad_token_id=tokenizer.pad_token_id, seed=SEED)

needed = cfg.num_steps * cfg.grad_accum * cfg.batch_size
print(f"Примеров: {len(tokenized)}, токенов с loss: {count_loss_tokens(tokenized)}")
print(f"За обучение будет показано {needed} примеров ≈ {needed / len(tokenized):.2f} эпохи")

## 3. Проверка патча

До обучения LoRA-адаптеры нулевые, модель совпадает с исходной. Loss на одном и том же фиксированном батче
в прогоне `hf` и в прогоне `cce` должен совпасть (сравнение — в последнем разделе).

In [ ]:
from finetune import evaluate_loss

fixed_batch = next(iter(make_train_dataloader(tokenized, batch_size=cfg.batch_size,
                                              pad_token_id=tokenizer.pad_token_id, shuffle=False)))
init_loss = evaluate_loss(model, fixed_batch, device=DEVICE)
(RESULTS / f"finetune_{LOSS_IMPL}_init_loss.json").write_text(json.dumps({"init_loss": init_loss}))
print(f"{LOSS_IMPL}: loss до обучения = {init_loss:.6f}")

## 4. Обучение

In [ ]:
from finetune import train

log = train(model, dataloader, cfg, device=DEVICE)
log.tail()

## 5. Максимальный батч (необязательно, аналог рис. 1)

Удвоением и двоичным поиском ищем наибольший батч фиксированной длины, который помещается в память.
Запускать после обучения: при поиске будут намеренные OOM.

In [ ]:
from finetune import find_max_batch_size, make_random_batch_factory

SEQ_LEN = 512
make_batch = make_random_batch_factory(vocab_size=model.config.vocab_size, seq_len=SEQ_LEN)
max_bs = find_max_batch_size(model, make_batch, start=1, limit=256, device=DEVICE)
(RESULTS / f"finetune_{LOSS_IMPL}_max_batch.json").write_text(json.dumps({"seq_len": SEQ_LEN, "max_batch": max_bs}))
print(f"{LOSS_IMPL}: максимальный батч при длине {SEQ_LEN} = {max_bs} ({max_bs * SEQ_LEN} токенов)")

## 6. Сравнение прогонов (читает только файлы из results/)

In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text()) if path.exists() else {}


runs = {impl: pd.read_csv(RESULTS / f"finetune_{impl}.csv")
        for impl in ("hf", "cce") if (RESULTS / f"finetune_{impl}.csv").exists()}

fig, ax = plt.subplots(figsize=(10, 4))
for impl, df in runs.items():
    line, = ax.plot(df["step"], df["loss"], alpha=0.25)
    ax.plot(df["step"], df["loss"].rolling(20, min_periods=1).mean(), color=line.get_color(),
            label=f"{impl} (скользящее среднее, 20 шагов)")
ax.set_xlabel("шаг оптимизатора")
ax.set_ylabel("training loss")
ax.set_title("LoRA fine-tuning: HF loss против CCE")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "finetune_loss_curves.png", dpi=150)

In [ ]:
summary = pd.DataFrame({
    impl: {
        "init_loss": read_json(RESULTS / f"finetune_{impl}_init_loss.json").get("init_loss"),
        "loss_last50_mean": df["loss"].tail(50).mean(),
        "peak_mem_mb_max": df["peak_mem_mb"].max(),
        "tokens_per_s_median": df["tokens_per_s"].iloc[5:].median(),   # первые шаги — прогрев
        "step_time_s_median": df["step_time_s"].iloc[5:].median(),
        "max_batch": read_json(RESULTS / f"finetune_{impl}_max_batch.json").get("max_batch"),
    }
    for impl, df in runs.items()
}).T
summary.to_csv(RESULTS / "finetune_summary.csv")

if {"hf", "cce"} <= runs.keys():
    diff = (runs["cce"]["loss"] - runs["hf"]["loss"]).abs()
    print(f"|loss_cce − loss_hf| по шагам: среднее {diff.mean():.4f}, максимум {diff.max():.4f}")
summary